In [2]:
!pip install faiss-cpu

In [3]:
## Imports and API test

import os
import faiss
from dotenv import load_dotenv
from openai import OpenAI
import json
import fitz  # PyMuPDF
import numpy as np
import openpyxl

In [4]:
# Load environment

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Say hello"}],
    max_tokens=10
)
print("OpenAI connected:", response.choices[0].message.content)

OpenAI connected: Hello! How can I assist you today?


In [5]:
## Extract text from eCrash User Guide PDF

pdf_path = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\LA eCrash User Guide.pdf"
doc = fitz.open(pdf_path)

pages = []
for i, page in enumerate(doc):
    text = page.get_text().strip()
    if len(text) > 50:
        pages.append({"page": i + 1, "text": text})

doc.close()
print(f"Extracted {len(pages)} pages with content")

Extracted 52 pages with content


In [6]:
## Chunk the pages into sections for ChromaDB

chunks = []
for i in range(len(pages)):
    text = pages[i]["text"]
    if i + 1 < len(pages):
        text += "\n" + pages[i + 1]["text"]
    
    chunk_id = f"page_{pages[i]['page']}"
    chunks.append({
        "id": chunk_id,
        "text": text[:3000],
        "page": pages[i]["page"]
    })

print(f"Created {len(chunks)} chunks")

Created 52 chunks


In [7]:
# Embed all chunks
all_texts = [chunk["text"] for chunk in chunks]
all_embeddings = []

batch_size = 10
for i in range(0, len(all_texts), batch_size):
    batch = all_texts[i:i + batch_size]
    
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    
    for item in response.data:
        all_embeddings.append(item.embedding)
    
    print(f"Batch {i // batch_size + 1}/{(len(all_texts) // batch_size) + 1} done")

# Convert to numpy array
embedding_matrix = np.array(all_embeddings, dtype="float32")

# Build FAISS index
dimension = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

# Save FAISS index and chunk metadata
faiss.write_index(index, r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\ecrash_faiss.index")

metadata = [{"id": chunk["id"], "page": chunk["page"], "text": chunk["text"]} for chunk in chunks]
with open(r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\ecrash_metadata.json", "w") as f:
    json.dump(metadata, f)

print(f"\nDone! FAISS index built with {index.ntotal} vectors (dimension: {dimension})")

Batch 1/6 done
Batch 2/6 done
Batch 3/6 done
Batch 4/6 done
Batch 5/6 done
Batch 6/6 done

Done! FAISS index built with 52 vectors (dimension: 1536)


In [9]:
## Test RAG retrieval

def retrieve_relevant_chunks(query, top_k=3):
    # Embed the query
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query]
    )
    query_embedding = np.array([response.data[0].embedding], dtype="float32")
    
    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for idx in indices[0]:
        results.append(metadata[idx])
    
    return results

# Test: search for "manner of crash" coding rules
results = retrieve_relevant_chunks("manner of crash coding rules")

print("Query: 'manner of crash coding rules'\n")
for i, r in enumerate(results):
    print(f"Result {i+1} (page {r['page']}):")
print(f"{r['text'][:200]}\n")

Query: 'manner of crash coding rules'

Result 1 (page 21):
Result 2 (page 22):
Result 3 (page 23):
Louisiana eCrash User Guide & Report Manual
93 / 393
Code
Description
102
Angle - Left into Flow
103
Angle - Right into Flow
104
Angle - Right Overtake
105
Angle - Perpendicular/Other
Angle
200
Front 



In [10]:
## Read a narrative from a Word file

from docx import Document

narratives_dir = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\narratives"

# Read one narrative as a test
test_file = os.listdir(narratives_dir)[0]
doc = Document(os.path.join(narratives_dir, test_file))
narrative_text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

case_number = test_file.replace(".docx", "")

print(f"Case Number: {case_number}")
print(f"\nNarrative:\n{narrative_text}")

Case Number: 2025018963

Narrative:
2025018963
ON SATURDAY, FEBRUARY 22, 2025, AT APPROXIMATELY 10:30 PM, OFFICER NICHOLAS SMITH, MANNING UNIT 523D, OF THE FIFTH DSITRCIT "D" PLATOON ERECEIVED A CALL TO INVESTIGATE A SIGNAL 20-I, RELATIVE TO AN ACCIDENT WITH INJURY, AT SPAIN / NORTH ROBERTSON.
OFFICER SMITH SPOKE WITH DRIVER #1 WHO STATED THE FOLLOWING. DRIVER #1 STATE SHE WAS TRAVELLING NORTHBOUND ON SPAIN ST AND "JUST FOLLOWING MY GPS" AND WHILE CROSSING N ROBERTSON ST STRUCK VEHICLE #2. DRIVER #2 STATED HER NECK HURT AFTER THE INCIDENT.
OFFICER SMITH SPOKE WITH DRIVER #2 WHO STATED THE FOLLOWING. DRIVER #2 STATED SHE WAS TRAVELLING EASTBOUND ON NORTH ROBERTSON ST WHEN VEHICLE #1 STRUCK THE SIDE OF VEHICLE #2 AT THE INTERSECTION OF SPAIN ST. DRIVER #2 STATED THAT BOTH HER LEGS HURT.
OFFICER SMITH SPOKE WITH ALL THE PASSENGERS WHO ADVISED ONLY PASSENGER #2.4 HAD AN INJURED TOW, HOWEVER ALL OTHER PARTIES WERE NOT INJURED.
OFFICER SMITH SPOKE WITH THE OWNER OF 1503 SPAIN ST WHHO ADVISED

In [12]:
## Pydantic schema validation

from pydantic import BaseModel, Field
from typing import Optional


class CrashPrediction(BaseModel):
    state_case_number: str
    manner_of_crash: str
    first_harmful_event: str
    v1_maneuver: str
    v2_maneuver: Optional[str] = None
    v3_maneuver: Optional[str] = None
    v4_maneuver: Optional[str] = None
    reasoning: str

print("CrashPrediction schema defined")

CrashPrediction schema defined


In [14]:
## Build the system prompt and RAG-augmented query

def interpret_narrative(narrative, case_number):
    rag_results = retrieve_relevant_chunks("manner of crash first harmful event vehicle maneuver coding rules", top_k=5)
    coding_rules = "\n\n---\n\n".join([r["text"] for r in rag_results])
    
    system_prompt = """You are a Louisiana crash report coding specialist trained on the eCrash User Guide.
Given a crash narrative, you must classify the crash by assigning official codes to these fields:

1. Manner of Crash (MOC): How the vehicles came together
2. First Harmful Event (FHE): The first event that caused injury or damage
3. Vehicle Maneuver: What each vehicle was doing before the crash

CRITICAL RULES FOR PARKED vs STOPPED:
- 500 Parked = vehicle is in a designated parking area, parking lot, or off the roadway. NOT in transport.
- 501 Stopped = vehicle is stopped in a TRAVEL LANE (at a light, in traffic, broken down in the road, or any part of the vehicle/trailer occupying a travel lane). IS in transport.
- THE LOCATION DETERMINES THE CODE: If any part of the vehicle or its attached trailer is in a travel lane, the vehicle is STOPPED (501), not PARKED (500), even if the driver says they "parked."
- A vehicle that is STOPPED (501) in a travel lane IS considered "in transport" for MOC purposes.
- A vehicle that is PARKED (500) off the roadway is NOT "in transport" for MOC purposes.

CRITICAL RULES FOR MANNER OF CRASH (MOC) - POINT OF CONTACT:
- MOC is determined by the ACTUAL POINT OF CONTACT between vehicles, NOT by the direction of approach.
- 300 Front to rear (rear end) = the FRONT of one vehicle strikes the REAR of another vehicle. BOTH vehicles must be facing the same direction.
- 505 Sideswipe same direction = the SIDE of one vehicle contacts the SIDE of another, both traveling same direction. This includes hitting an open door, a side panel, or the side of a trailer.
- 105 Angle perpendicular/other angle = vehicles are at an angle to each other when they collide. This includes: a vehicle perpendicular to traffic (e.g., pulling into a driveway) being struck by a vehicle on the road, or any collision where the vehicles are NOT parallel.
- If a vehicle is perpendicular to traffic flow (turning into a driveway, crossing lanes) and another vehicle strikes it, this is an ANGLE collision (100-105), NOT a rear-end (300).
- If a vehicle strikes the SIDE (including door) of another vehicle while both travel the same direction, this is a SIDESWIPE (505), NOT a rear-end (300).
- 300 Rear end is ONLY correct when the front of one vehicle directly strikes the rear bumper/end of another vehicle, and both are facing the same direction.

RULES FOR FIRST HARMFUL EVENT (FHE):
- If a moving vehicle strikes another vehicle that is "in transport" (including stopped in travel lane), FHE = 201
- If a moving vehicle strikes a PARKED vehicle (off roadway), FHE = 202
- The FHE must be consistent with the MOC and vehicle statuses

RULES FOR VEHICLE MANEUVER:
- Determine what each vehicle was doing IMMEDIATELY BEFORE the crash, not during the crash.
- 100 Going straight = vehicle was traveling in a straight line on the roadway
- 106 Turning left = vehicle was actively making a left turn at the moment of impact
- 107 Turning right = vehicle was actively making a right turn at the moment of impact
- Do NOT code a vehicle as turning unless the narrative clearly states the vehicle was in the act of turning when the crash occurred. If the vehicle had already completed a turn and was traveling straight, code as 100 Going straight.

IMPORTANT:
- Use ONLY the official code values listed below.
- Output the code number AND description (e.g., "300~Front to rear - rear end")
- If a "vehicle" is actually an unattached trailer, property, or non-motorized object, output "DELETE" for that vehicle
- Output valid JSON only, no extra text

CODE VALUES:

Manner of Crash (MOC):
000~Not a collision between two motor vehicles in transport
100~Angle - left overtake
101~Angle - left opposite direction
102~Angle - left into flow
103~Angle - right into flow
104~Angle - right overtake
105~Angle - perpendicular/other angle
200~Front to front - head on
300~Front to rear - rear end
400~Backing - rear to front
401~Backing - rear to rear
402~Backing - rear to side
500~Angle - left across flow
501~Angle - right across flow
502~Sideswipe - opposite direction
505~Sideswipe - same direction
980~Other
999~Unknown

First Harmful Event (FHE):
100~Cargo/equipment loss or shift
101~Fell/jumped from motor vehicle
102~Fire/explosion
103~Immersion, full or partial
104~Jackknife
105~Overturn/rollover
106~Thrown or falling object
198~Other non-collision harmful event
200~Collision with animal (live)
201~Collision with motor vehicle in transport
202~Collision with parked motor vehicle
203~Collision with pedalcycle
204~Collision with pedestrian
205~Collision with railway vehicle
206~Collision with object at rest from MV in transport
207~Collision with falling, shifting cargo
208~Collision with work zone/maintenance equipment
209~Collision with farm equipment
297~Collision with other non-motorist
298~Collision with other non-fixed object
300~Collision with bridge overhead structure
301~Collision with bridge pier or support
302~Collision with bridge rail
303~Collision with cable barrier
304~Collision with concrete traffic barrier
305~Collision with culvert
306~Collision with curb
307~Collision with ditch
308~Collision with embankment
309~Collision with fence
310~Collision with guardrail end terminal
311~Collision with guardrail face
312~Collision with impact attenuator/crash cushion
313~Collision with mailbox
314~Collision with traffic sign support
315~Collision with traffic signal support
316~Collision with tree (standing)
317~Collision with utility pole/light support
396~Collision with other post, pole, or support
397~Collision with other traffic barrier
398~Collision with other fixed object
399~Collision with unknown fixed object

Vehicle Maneuver:
100~Going straight
101~Backing
102~Merging
103~Making U-turn
104~Negotiating a curve
106~Turning left
107~Turning right
108~Traveling wrong way
200~Leaving a parking position
400~Slowing
500~Parked
501~Stopped
980~Other
999~Unknown"""

    user_prompt = f"""## Relevant Coding Rules from eCrash User Guide:
{coding_rules}

## Crash Narrative (State Case #{case_number}):
{narrative}

Think step by step:
1. How many vehicles are involved?
2. For EACH vehicle: Is any part of it (including attached trailers) in a travel lane? If yes = in transport. If in a parking lot or off the roadway = parked, not in transport.
3. What was each vehicle doing IMMEDIATELY BEFORE the crash?
4. What was the FIRST harmful event?
5. What is the ACTUAL POINT OF CONTACT? (front-to-rear, side-to-side, perpendicular angle?) The point of contact determines the MOC, not the direction of approach.

Then output ONLY valid JSON in this format:
{{
  "state_case_number": "{case_number}",
  "manner_of_crash": "CODE~DESCRIPTION",
  "first_harmful_event": "CODE~DESCRIPTION",
  "v1_maneuver": "CODE~DESCRIPTION",
  "v2_maneuver": "CODE~DESCRIPTION or DELETE or null",
  "v3_maneuver": "CODE~DESCRIPTION or DELETE or null",
  "v4_maneuver": "CODE~DESCRIPTION or DELETE or null",
  "reasoning": "Your step-by-step reasoning here"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    
    raw_result = json.loads(response.choices[0].message.content)
    
    # Validate with Pydantic
    try:
        validated = CrashPrediction(**raw_result)
        print(f"  ✅ Pydantic validation passed")
        return validated.model_dump()
    except Exception as e:
        print(f"  ❌ Validation error: {e}")
        return raw_result

In [15]:
## Process narratives (testing - limited to 2)

from docx import Document

narratives_dir = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\narratives"

all_results = []

# Get only the first 2 files for testing
files_to_process = [f for f in sorted(os.listdir(narratives_dir)) 
                    if f.endswith(".docx") and not f.startswith("~$")][:2]

for filename in files_to_process:
    filepath = os.path.join(narratives_dir, filename)
    doc = Document(filepath)
    narrative = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    case_number = filename.replace(".docx", "")
    
    print(f"\nProcessing: {case_number}...")
    
    result = interpret_narrative(narrative, case_number)
    all_results.append(result)
    
    print(f"  MOC: {result['manner_of_crash']}")
    print(f"  FHE: {result['first_harmful_event']}")
    print(f"  V1:  {result['v1_maneuver']}")
    print(f"  V2:  {result.get('v2_maneuver', 'N/A')}")
    print(f"  V3:  {result.get('v3_maneuver', 'N/A')}")

print(f"\n\nDone! Processed {len(all_results)} narratives.")


Processing: 2025018963...
  ✅ Pydantic validation passed
  MOC: 105~Angle - perpendicular/other angle
  FHE: 201~Collision with motor vehicle in transport
  V1:  100~Going straight
  V2:  100~Going straight
  V3:  None

Processing: 2025020863...
  ✅ Pydantic validation passed
  MOC: 000~Not a collision between two motor vehicles in transport
  FHE: 202~Collision with parked motor vehicle
  V1:  100~Going straight
  V2:  500~Parked
  V3:  500~Parked


Done! Processed 2 narratives.


In [16]:
## Save results to JSON file

import json

output_path = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\predictions.json"

with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved {len(all_results)} predictions to predictions.json")

Saved 2 predictions to predictions.json


In [18]:


predictions_path = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\State_number.xlsx"
ground_truth_path = r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\Crash.xlsx"

# --- Load predictions (headers on row 2, data starts row 3) ---
wb_p = openpyxl.load_workbook(predictions_path, read_only=True)
ws_p = wb_p.active
headers_p = {cell.value: cell.column for cell in ws_p[2] if cell.value is not None}

predictions = {}
for row_num in range(3, ws_p.max_row + 1):
    case = ws_p.cell(row=row_num, column=headers_p["State Case Number"]).value
    if case:
        predictions[str(case).strip()] = {
            "moc": ws_p.cell(row=row_num, column=headers_p["Correct Manner Code"]).value,
            "fhe": ws_p.cell(row=row_num, column=headers_p["First Harmful Event(FHE)"]).value,
            "v1":  ws_p.cell(row=row_num, column=headers_p["V1 Correct Maneuver"]).value,
            "v2":  ws_p.cell(row=row_num, column=headers_p["V2 Correct Maneuver"]).value,
            "v3":  ws_p.cell(row=row_num, column=headers_p["V3 Correct Maneuver"]).value,
            "v4":  ws_p.cell(row=row_num, column=headers_p["V4 Correct Maneuver"]).value,
        }
wb_p.close()

# --- Load ground truth (headers on row 1, data starts row 2) ---
wb_g = openpyxl.load_workbook(ground_truth_path, read_only=True)
ws_g = wb_g.active
headers_g = {cell.value: cell.column for cell in ws_g[1] if cell.value is not None}

ground_truth = {}
for row_num in range(2, ws_g.max_row + 1):
    case = ws_g.cell(row=row_num, column=headers_g["State Case Number"]).value
    if case:
        ground_truth[str(case).strip()] = {
            "moc": ws_g.cell(row=row_num, column=headers_g["Correct Manner Code"]).value,
            "fhe": ws_g.cell(row=row_num, column=headers_g["First Harmful Event(FHE)"]).value,
            "v1":  ws_g.cell(row=row_num, column=headers_g["V1 Correct Maneuver"]).value,
            "v2":  ws_g.cell(row=row_num, column=headers_g["V2 Correct Maneuver"]).value,
            "v3":  ws_g.cell(row=row_num, column=headers_g["V3 Correct Maneuver"]).value,
            "v4":  ws_g.cell(row=row_num, column=headers_g["V4 Correct Maneuver"]).value,
        }
wb_g.close()

# --- Compare predictions to ground truth ---
def normalize(val):
    """Normalize values for comparison: '~', None, 'null', 'DELETE' all become 'N/A'."""
    if val is None or str(val).strip() in ["~", "null", "None", ""]:
        return "N/A"
    return str(val).strip()

fields = ["moc", "fhe", "v1", "v2", "v3", "v4"]
field_correct = {f: 0 for f in fields}
field_total = {f: 0 for f in fields}
case_results = []

for case, gt in ground_truth.items():
    if case not in predictions:
        continue
    
    pred = predictions[case]
    case_correct = 0
    case_total = 0
    field_status = {}
    
    for f in fields:
        gt_val = normalize(gt[f])
        pred_val = normalize(pred[f])
        
        # Skip fields where both are N/A (no vehicle)
        if gt_val == "N/A" and pred_val == "N/A":
            field_status[f] = "—"
            continue
        
        field_total[f] += 1
        case_total += 1
        
        if gt_val == pred_val:
            field_correct[f] += 1
            case_correct += 1
            field_status[f] = "✅"
        else:
            field_status[f] = f"❌ pred:{pred_val} | gt:{gt_val}"
    
    case_results.append({
        "case": case,
        "correct": case_correct,
        "total": case_total,
        "fields": field_status
    })

# --- Print results ---
print("=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

for r in case_results:
    print(f"\nCase {r['case']}: {r['correct']}/{r['total']} fields correct")
    for f, status in r["fields"].items():
        print(f"  {f.upper():4} {status}")

print("\n" + "=" * 70)
print("FIELD-LEVEL ACCURACY")
print("=" * 70)
total_correct = 0
total_fields = 0
for f in fields:
    if field_total[f] > 0:
        acc = field_correct[f] / field_total[f] * 100
        print(f"  {f.upper():4}: {field_correct[f]}/{field_total[f]} = {acc:.1f}%")
        total_correct += field_correct[f]
        total_fields += field_total[f]

if total_fields > 0:
    overall = total_correct / total_fields * 100
    print(f"\n  OVERALL: {total_correct}/{total_fields} = {overall:.1f}%")
print(f"\n  Cases evaluated: {len(case_results)}")

EVALUATION RESULTS

Case 2025020863: 5/5 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   ✅
  V4   —

Case 2025029552: 5/5 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   ✅
  V4   —

Case 2025044701: 5/5 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   ✅
  V4   —

Case 2025043720: 3/4 fields correct
  MOC  ❌ pred:300~Front to rear - rear end | gt:105~Angle - perpendicular/other angle
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   —
  V4   —

Case 2025075212: 4/4 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   —
  V4   —

Case 2025018963: 4/4 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   —
  V4   —

Case 2025076534: 2/4 fields correct
  MOC  ❌ pred:105~Angle - perpendicular/other angle | gt:505~Sideswipe - same direction
  FHE  ✅
  V1   ❌ pred:501~Stopped | gt:100~Going straight
  V2   ✅
  V3   —
  V4   —

Case 2025050402: 4/4 fields correct
  MOC  ✅
  FHE  ✅
  V1   ✅
  V2   ✅
  V3   —
  V4   —

Case 2025096800: 4/4 fields correct
  MOC  ✅
  FHE  ✅
 

In [19]:


# Get cases from predictions file
wb_p = openpyxl.load_workbook(r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\State_number.xlsx", read_only=True)
ws_p = wb_p.active
headers_p = {cell.value: cell.column for cell in ws_p[2] if cell.value is not None}
pred_cases = set()
for row_num in range(3, ws_p.max_row + 1):
    val = ws_p.cell(row=row_num, column=headers_p["State Case Number"]).value
    moc = ws_p.cell(row=row_num, column=headers_p["Correct Manner Code"]).value
    if val:
        pred_cases.add((str(val).strip(), bool(moc)))
wb_p.close()

# Get cases from ground truth
wb_g = openpyxl.load_workbook(r"C:\Users\Hamidreza\OneDrive\Desktop\LLM\LLM_Project\Crash.xlsx", read_only=True)
ws_g = wb_g.active
headers_g = {cell.value: cell.column for cell in ws_g[1] if cell.value is not None}
gt_cases = set()
for row_num in range(2, ws_g.max_row + 1):
    val = ws_g.cell(row=row_num, column=headers_g["State Case Number"]).value
    if val:
        gt_cases.add(str(val).strip())
wb_g.close()

print(f"Cases in predictions file: {len(pred_cases)}")
for c, has_pred in sorted(pred_cases):
    status = "✅ predicted" if has_pred else "❌ NO PREDICTION"
    in_gt = "(in GT)" if c in gt_cases else "(NOT in GT)"
    print(f"  {c} {status} {in_gt}")

print(f"\nCases in GT but NOT in predictions:")
pred_set = {c for c, _ in pred_cases}
for c in sorted(gt_cases - pred_set):
    print(f"  {c}")

Cases in predictions file: 15
  2025018963 ✅ predicted (in GT)
  2025020863 ✅ predicted (in GT)
  2025029552 ✅ predicted (in GT)
  2025043720 ✅ predicted (in GT)
  2025044701 ✅ predicted (in GT)
  2025050402 ✅ predicted (in GT)
  2025075212 ✅ predicted (in GT)
  2025075248 ✅ predicted (in GT)
  2025076534 ✅ predicted (in GT)
  2025096800 ✅ predicted (in GT)
  2025096801 ✅ predicted (NOT in GT)
  2025108122 ✅ predicted (in GT)
  2025123078 ✅ predicted (in GT)
  2025123969 ✅ predicted (in GT)
  2025125093 ✅ predicted (in GT)

Cases in GT but NOT in predictions:
  2025000214
  2025000879
  2025001699
  2025005054
  2025005127
  2025011275
  2025011451
  2025013760
  2025017596
  2025017835
  2025021427
  2025025722
  2025025808
  2025026543
  2025030723
  2025034331
  2025047343
  2025047936
  2025048980
  2025049094
  2025068793
  2025076133
  2025076783
  2025077952
  2025078563
  2025079775
  2025081350
  2025081788
  2025094435
  2025108116
  2025108551
  2025108834
  2025109959
  202